## 1- Imports

In [ ]:
!pip install torch==2.4.1 --index-url https://download.pytorch.org/whl/cu121
!pip install torch_cluster==1.6.3+pt24cu121 --find-links https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install torch_geometric==2.5.3
!pip install pyg-lib==0.4.0+pt24cu121 --find-links https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install torch_scatter==2.1.2+pt24cu121 --find-links https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install torch_sparse==0.6.18+pt24cu121 --find-links https://data.pyg.org/whl/torch-2.4.0+cu121.html
!pip install torch_spline_conv==1.2.2+pt24cu121 --find-links https://data.pyg.org/whl/torch-2.4.0+cu121.html

In [ ]:
!pip install fairchem-core==1.10.0

In [ ]:
!git clone https://github.com/hoang-hoa-tham-hs/fairchem-tio2-s2ef.git
!pip install ase lmdb wandb numba

In [ ]:
%cd /content/fairchem-tio2-s2ef

In [ ]:
import sys
import torch
import os
# sys.path.append(r"D:\Train_thử\fairchem-tio2-s2ef")

from ocpmodels.trainers import ForcesTrainer
from ocpmodels import models
from ocpmodels.common import logger
from ocpmodels.common.utils import setup_logging

from google.colab import drive
drive.mount('/content/drive')

setup_logging()

In [ ]:
# a simple sanity check that a GPU is available
if torch.cuda.is_available():
    print("True")
else:
    print("False")

torch.cuda.set_device(0)

## The essential steps for training an OCP model
1. Download data

2. Preprocess data (if necessary)

3. Define or load a configuration (config), which includes the following

- task
- model
- optimizer
- dataset
- trainer
4. Train

5. Evaluate

6. Predict

## 2. Dataset

In [ ]:
from ocpmodels.datasets import TrajectoryLmdbDataset, OC22LmdbDataset
import numpy as np
import copy

# train_src = r"D:\Train_thử\fairchem-tio2-s2ef\tio2_s2ef\data\tio2_lmdb\200k"

# train_dataset = TrajectoryLmdbDataset({"src": train_src})

train_src = r"/content/drive/MyDrive/abc/tio2_filtered/train"
val_src   = r"/content/drive/MyDrive/abc/tio2_filtered/val"
test_src  = r"/content/drive/MyDrive/abc/tio2_filtered/test"

# train_dataset = OC22LmdbDataset({"src": train_src})

mean  = 0.0
stdev = 25.119809935106424

# energies = []
# for data in train_dataset:
#   energies.append(data.y)

# mean = np.mean(energies)
# stdev = np.std(energies)

## 3. Define config

## 3.1 Task

In [ ]:
task = {
    "dataset": "oc22_lmdb",                          # MUST be oc22_lmdb for OC22
    "description": "Regressing to energies and forces for OC22 S2EF task",
    "type": "regression",
    "metric": "mae",
    "labels": ["potential energy"],
    "grad_input": "atomic forces",
    "train_on_free_atoms": True,
    "eval_on_free_atoms": True,
}

## 3.2 Model

In [ ]:
model = {
    "name": "gemnet_t",                              # model architecture
    "num_spherical": 7,
    "num_radial": 128,
    "num_blocks": 3,
    "emb_size_atom": 512,
    "emb_size_edge": 512,
    "emb_size_trip": 64,
    "emb_size_rbf": 16,
    "emb_size_cbf": 16,
    "emb_size_bil_trip": 64,
    "num_before_skip": 1,
    "num_after_skip": 2,
    "num_concat": 1,
    "num_atom": 3,
    "cutoff": 6.0,
    "max_neighbors": 50,
    "rbf": {"name": "gaussian"},
    "envelope": {
        "name": "polynomial",
        "exponent": 5,
    },
    "cbf": {"name": "spherical_harmonics"},
    "extensive": True,
    "otf_graph": True,
    "output_init": "HeOrthogonal",
    "activation": "silu",
    "scale_file": "/content/fairchem-tio2-s2ef/configs/s2ef/all/gemnet/scaling_factors/gemnet-dT.json",
    "regress_forces": True,
    "direct_forces": True,
}

## 3.3 Optimizer

In [ ]:
optimizer = {
    "batch_size": 4,                                 # lower if GPU runs out of memory
    "eval_batch_size": 4,
    "num_workers": 0,                                # lower to 0 if DataLoader freezes
    "lr_initial": 1.0e-5,
    "optimizer": "AdamW",
    "optimizer_params": {"amsgrad": True},
    "scheduler": "ReduceLROnPlateau",
    "mode": "min",
    "factor": 0.8,
    "patience": 3,
    "max_epochs": 20,                                # how many epochs to train
    "force_coefficient": 100,                        # weight of force loss vs energy loss
    "ema_decay": 0.999,
    "clip_grad_norm": 10,
    "loss_energy": "mae",
    "loss_force": "l2mae",
}

## 3.4 Dataset

In [ ]:
# Dataset
dataset = [
  {'src': train_src,
   'normalize_labels': True,
   "target_mean": mean,
   "target_std": stdev,
   "grad_target_mean": 0.0,
   "grad_target_std": stdev
   },
  {'src': val_src},
  {'src': test_src}
]

## 3.5 Trainer

Use the ForcesTrainer for the S2EF and IS2RS tasks, and the EnergyTrainer for the IS2RE task

In [ ]:
trainer = ForcesTrainer(
    task=task,
    model=copy.deepcopy(model),
    dataset=dataset,
    optimizer=optimizer,
    identifier="OC22-S2EF-TiO2",
    run_dir="/content",                                    # saves checkpoints/logs/results here
    is_debug=False,                                  # False = saves checkpoints and logs
    print_every=5,                                   # print loss every N steps
    seed=0,
    logger="tensorboard",                            # "tensorboard" or "wandb"
    local_rank=0,
    amp=False,                                        # mixed precision (set False if issues)
    cpu=False,
)

## 3.6 Load pretrained model

In [ ]:
# Verify model/Checking model
from fairchem.core.models.model_registry import available_pretrained_models
models = available_pretrained_models

keyword = "GemNet"

filtered = [m for m in models if keyword in m]

print(filtered)

In [ ]:
# Get Checkpoint
from fairchem.core.models.model_registry import model_name_to_local_file
checkpoint_path = model_name_to_local_file(
    "GemNet-dT-S2EFS-OC22",
    local_cache="/content/checkpoints"
)

# Load checkpoint
trainer.load_checkpoint(checkpoint_path=checkpoint_path)

## 3.7 Freeze layer

In [ ]:
# After trainer.load_checkpoint(), freeze layers selectively
for name, param in trainer.model.named_parameters():
    # Freeze everything except output blocks
    if "out_blocks" not in name and "out_mlp" not in name:
        param.requires_grad = False

# Verify what's trainable
trainable = [n for n, p in trainer.model.named_parameters() if p.requires_grad]
print(f"Trainable layers: {trainable}")

## 3.8 Check the model

In [ ]:
import torch
print(torch.cuda.is_available())       # must be True
param = next(trainer.model.parameters())
print(param.device)                    # must show cuda:0

## 4. Train GemNet

In [ ]:
trainer.train()

## 5. Evaluate

## 5.1 Load pretrained and Fine-tune model

In [ ]:
# Pretrain model
pretrained = ForcesTrainer(
    task=task,
    model=copy.deepcopy(model),
    dataset=dataset,
    optimizer=optimizer,
    identifier="OC22-S2EF-TiO2",
    run_dir="/content",
    is_debug=False,
    print_every=5,
    seed=0,
    logger="tensorboard",
    local_rank=0,
    amp=False,
    cpu=False,
)

In [ ]:
# Pretrained model checkpoint
from fairchem.core.models.model_registry import model_name_to_local_file

checkpoint_path = model_name_to_local_file(
    "GemNet-dT-S2EFS-OC22",
    local_cache="/content/checkpoints"
)

# Load pretrained model
pretrained.load_checkpoint(checkpoint_path=checkpoint_path)

In [ ]:
# Fine tune checkpoint
fine_tune_checkpoint_path = '/content/drive/MyDrive/abc/best_checkpoint.pt'

# Load fine tune model
trainer.load_checkpoint(checkpoint_path=fine_tune_checkpoint_path)

## 5.2 Predict

In [ ]:
# Fine tune model make predictions on the existing test_loader
predictions = trainer.predict(trainer.test_loader, results_file="s2ef_results", disable_tqdm=False)

# Pretrained model make predictions on the existing test_loader
predictions_raw = pretrained.predict(pretrained.test_loader, results_file="s2ef_results_raw", disable_tqdm=False)

## 5.3 Evaluation Metrics

## 5.3.1 Training Logs

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/abc

## 5.3.2 Metrics

MAE

In [ ]:
# fINE TUNE MODEL PREDICTIONS
import numpy as np

# Load data
results_path = "/s2ef_s2ef_results_raw.npz"

data = np.load(results_path, allow_pickle=True)
print(data.files)

# Calculate MAE
pred_energy = data["energy"]
ids = data["ids"]

true_energy = []

for d in trainer.test_loader.dataset:
    true_energy.append(d.y)

true_energy = np.array(true_energy)

mae = np.mean(np.abs(pred_energy - true_energy))

print("Energy MAE:", mae)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# PRETRAINED MODEL PREDICTIONS
import numpy as np

results_path = "/s2ef_s2ef_results.npz"

data = np.load(results_path, allow_pickle=True)
print(data.files)

pred_energy = data["energy"]
ids = data["ids"]

true_energy = []

for d in trainer.test_loader.dataset:
    true_energy.append(d.y)

true_energy = np.array(true_energy)

mae = np.mean(np.abs(pred_energy - true_energy))

print("Energy MAE:", mae)

In [ ]:
# FINE TUNE MODEL PREDICTIONS
import numpy as np

# Load data
results_path = "/s2ef_s2ef_results_raw.npz" #Alter file raw or result file

data = np.load(results_path, allow_pickle=True)
print(data.files)

# Extract predictions
pred_energy = data["energy"]
pred_forces = data["forces"]
ids = data["ids"]
chunk_idx = data["chunk_idx"]

# Collect true values
true_energy = []
true_forces = []

for d in trainer.test_loader.dataset:
    true_energy.append(d.y)
    true_forces.append(d.force)  # forces stored in d.force for OCP datasets

true_energy = np.array(true_energy)
true_forces = np.concatenate(true_forces, axis=0)  # flatten across all structures

# Calculate Energy MAE
mae_energy = np.mean(np.abs(pred_energy - true_energy))
print("Energy MAE:", mae_energy)

# Calculate Forces MAE
mae_forces = np.mean(np.abs(pred_forces - true_forces))
print("Forces MAE:", mae_forces)

# Per-component Forces MAE (x, y, z)
mae_forces_x = np.mean(np.abs(pred_forces[:, 0] - true_forces[:, 0]))
mae_forces_y = np.mean(np.abs(pred_forces[:, 1] - true_forces[:, 1]))
mae_forces_z = np.mean(np.abs(pred_forces[:, 2] - true_forces[:, 2]))
print(f"Forces MAE (x, y, z): {mae_forces_x:.6f}, {mae_forces_y:.6f}, {mae_forces_z:.6f}")

# Summary
print("\n=== Summary ===")
print(f"Total structures evaluated: {len(pred_energy)}")
print(f"Energy MAE: {mae_energy:.6f}")
print(f"Forces MAE: {mae_forces:.6f}")